In [8]:
import csv
import numpy as np
from tqdm import tqdm
import trimesh
from scipy.spatial import cKDTree

# --- CONFIG ---
gro_path = "../../data/npt-HK4.gro"               
sphere_radius_scale = 1.5                 # balls (atoms): 2.0 × van der Waals radius   # idk why 1.5 :(
bond_radius = 0.1                        # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                         # atom sphere detail (2 is moderate)

### Import data and Setup

In [9]:
def parse_gro(path, atoms_per_mol=84):
    """
    Parse a .gro file with multiple molecules of equal atom count.

    Parameters
    ----------
    path : str
        Path to .gro file
    atoms_per_mol : int
        Number of atoms per molecule (default 84)

    Returns
    -------
    molecules : dict[int, list[tuple]]
        Dictionary of molecules:
          molecules[i] = [(atomname, coord), ...] for atom coords in Å
        where i = 1..N_molecules
    """
    with open(path, "r") as f:
        _title = f.readline()
        n = int(f.readline().strip())  # total number of atoms
        lines = [f.readline() for _ in range(n)]
        box_line = f.readline().strip()  # box line

    # --- parse atom coordinates ---
    n_mol = n // atoms_per_mol  # determine number of molecules using integer division
    if n % atoms_per_mol != 0:
        raise ValueError(f"Total atoms {n} not divisible by atoms_per_mol={atoms_per_mol}")

    molecules = {}
    for m in range(n_mol):
        start = m * atoms_per_mol
        end = start + atoms_per_mol
        mol_atoms = []
        for line in lines[start:end]:
            # parse atom name and coords
            atomname = line[10:15].strip()
            x = float(line[20:28]) * 10.0  # nm → Å
            y = float(line[28:36]) * 10.0
            z = float(line[36:44]) * 10.0
            mol_atoms.append((atomname, np.array([x, y, z], dtype=float)))
        molecules[m+1] = mol_atoms

    # --- parse box dimensions ---
    box_vals = [float(x) for x in box_line.split()]
    if len(box_vals) == 3:
        # orthorhombic box
        box = np.array(box_vals) * 10.0  # Å
    elif len(box_vals) == 9:
        # triclinic box: x, y, z vectors in nm
        box = np.array(box_vals).reshape(3, 3) * 10.0  # Å
    else:
        raise ValueError(f"Unexpected box format with {len(box_vals)} values")

    return molecules, box


def infer_element(atomname):
    """
    Infer the chemical element symbol from an atom name string.

    This function attempts to deduce the element type from a given atom name,
    handling common water aliases (e.g., OW, HW), typical atom name conventions,
    and capitalization rules. If the atom name is empty or unrecognized, it defaults to "C" (carbon).

    Parameters
    ----------
    atomname : str
        The atom name from a .gro or similar file.

    Returns
    -------
    element : str
        The inferred element symbol (e.g., "C", "O", "H", "Cl").
    """
    
    # common water aliases
    if atomname in ("OW", "HW", "HW1", "HW2"): 
        return "O" if atomname=="OW" else "H"
    
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])   
    
    # join() joins items in an iterable into one string, '' is specified as the separator.
    # isalpha() method returns True if all the characters are alphabet letters (a-z).

    if a == '': 
        return "C"

    if len(a) >= 2 and a[1].islower(): 
        return (a[0]+a[1]).capitalize()
    
    return a[0].upper()

In [10]:
# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

In [11]:
molecules, box = parse_gro(gro_path, atoms_per_mol=84)

### Build molecules models

In [12]:
def mic_vector(dx, box_length):
    """Return minimum-image displacement for vector dx under PBC."""
    return dx - np.rint(dx / box_length) * box_length


def mic_distance(a, b, box_length):
    """Return MIC distance between two 3D points a,b."""
    return np.linalg.norm(mic_vector(b - a, box_length))


def build_molecule_ballstick(coords, elements,
                             vdw, cov,
                             box_length,
                             sphere_radius_scale=0.3,
                             sphere_subdiv=2,
                             bond_radius=0.1):
    """
    Build a trimesh ball-and-stick model of a molecule under periodic boundary conditions.

    Atoms are represented as spheres (with van der Waals radii, scaled), and bonds as cylinders
    (if the minimum-image convention distance between atoms is less than 1.2 times the sum of their covalent radii).

    Parameters
    ----------
    coords : np.ndarray, shape (n_atoms, 3)
        Cartesian coordinates of atoms (in Å).
    elements : list of str
        List of element symbols for each atom.
    vdw : dict
        Dictionary mapping element symbols to van der Waals radii (in Å).
    cov : dict
        Dictionary mapping element symbols to covalent radii (in Å).
    box_length : float or np.ndarray
        Simulation box length(s) for periodic boundary conditions (in Å).
    sphere_radius_scale : float, optional
        Scaling factor for atom sphere radii (default: 0.3).
    sphere_subdiv : int, optional
        Number of subdivisions for atom sphere mesh detail (default: 2).
    bond_radius : float, optional
        Cylinder radius for bonds (default: 0.1 Å).

    Returns
    -------
    molecule : trimesh.Trimesh
        Combined mesh of the molecule (atoms as spheres, bonds as cylinders).
    """
    
    # radii arrays
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.77) for e in elements])

    meshes = []

    # --- Atoms as spheres (wrapped into primary box [0,L)) ---
    coords_wrapped = np.mod(coords, box_length)
    for pos, r in zip(coords_wrapped, vdw_r * sphere_radius_scale):
        sph = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=float(r))
        sph.apply_translation(pos)
        meshes.append(sph)

    # --- Bonds (MIC criterion with covalent radii) ---
    n = len(coords)
    for i in range(n):
        for j in range(i+1, n):
            d = mic_distance(coords[i], coords[j], box_length)
            thr = 1.2 * (cov_r[i] + cov_r[j])
            if d < thr:
                # Unwrap j relative to i
                disp = mic_vector(coords[j] - coords[i], box_length)
                pos_i = np.mod(coords[i], box_length)
                pos_j = pos_i + disp  # may fall outside box but correct bond vector
                seg = np.vstack((pos_i, pos_j))
                cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                segment=seg, sections=24)
                meshes.append(cyl)

    # --- Merge all into one mesh ---
    molecule = trimesh.util.concatenate(meshes)
    
    return molecule


In [13]:
def molecules_to_meshes(molecules, box,
                        vdw, cov,
                        sphere_radius_scale=0.3,
                        sphere_subdiv=2,
                        bond_radius=0.1):
    """
    Convert parsed molecules into trimesh meshes.

    Parameters
    ----------
    molecules : dict[int, list[tuple]]
        From parse_gro(): molecules[i] = [(atomname, coords), ...]
        coords must be in Å
    box : np.ndarray
        Simulation box (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    vdw, cov : dict
        Van der Waals and covalent radii
    sphere_radius_scale : float
        Scaling factor for atom radii
    sphere_subdiv : int
        Subdivisions for icosphere (mesh resolution)
    bond_radius : float
        Cylinder radius for bonds

    Returns
    -------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes keyed by mol_id
    """
    mol_meshes = {}
    
    # assume orthorhombic box for now
    if box.shape == (3,):
        box_length = box
    else:
        raise NotImplementedError("Triclinic box handling not yet implemented")

    for mol_id, atoms in molecules.items():
        elements = [infer_element(name) for name, _ in atoms]
        coords = np.vstack([pos for _, pos in atoms])  # (n_atoms, 3)

        mesh = build_molecule_ballstick(
            coords, elements, vdw, cov, box_length,
            sphere_radius_scale=sphere_radius_scale,
            sphere_subdiv=sphere_subdiv,
            bond_radius=bond_radius
        )
        mol_meshes[mol_id] = mesh

    return mol_meshes


In [14]:
# Build meshes for all molecules
mol_meshes = molecules_to_meshes(molecules, box, vdw, cov,
                                 sphere_radius_scale=sphere_radius_scale,
                                 sphere_subdiv=sphere_subdiv,
                                 bond_radius=bond_radius
                                 )

### Nearest neighbors candidates

In [25]:
def compute_centroids_and_radii_pbc(mol_meshes, box):
    """
    Compute periodic-boundary-condition (PBC) aware centroids and radii for a set of molecules.

    For each molecule mesh, this function:
      1. Selects a reference vertex (atom) as the origin.
      2. Unwraps all other vertices relative to this reference using the minimum-image convention (MIC),
         so that all atoms are locally unwrapped and contiguous in space.
      3. Computes the centroid (geometric center) of the unwrapped coordinates.
      4. Calculates the maximum MIC distance from the centroid to any vertex, defining the molecule's effective radius.

    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their corresponding trimesh mesh objects.
    box : array-like, shape (3,) or (3,3)
        Simulation box dimensions (in Å). Should be a 3-element array for orthorhombic boxes.

    Returns
    -------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å), unwrapped in PBC.
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å), defined as the maximum MIC distance
        from the centroid to any vertex in the molecule.
    """
    centroids, radii = {}, {}
    box = np.array(box, dtype=float)

    for mol_id, mesh in mol_meshes.items():
        verts = mesh.vertices
        ref = verts[0]  # reference atom
        disp = mic_vector(verts - ref, box)
        unwrapped = ref + disp

        # centroid in unwrapped space
        center = unwrapped.mean(axis=0)

        # MIC distances from centroid to each vertex
        disp_center = mic_vector(verts - center, box)
        radius = np.linalg.norm(disp_center, axis=1).max()

        centroids[mol_id] = center
        radii[mol_id] = radius

    return centroids, radii


In [71]:
for i, j in mol_meshes.items():
    print(i, j)

1 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
2 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
3 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
4 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
5 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
6 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
7 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
8 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
9 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
10 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
11 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
12 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
13 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
14 <trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>
1

In [26]:
def wrap_points(points, box):
    """
    Wrap points into the primary simulation box using periodic boundary conditions.

    Each coordinate of the input points is wrapped into the interval [0, box_length)
    by applying the modulo operation with respect to the box dimensions. This ensures
    that all points are mapped inside the simulation box, consistent with periodic boundary conditions.

    This is the inverse operation of mic_vector.

    Parameters
    ----------
    points : np.ndarray
        Array of points to wrap. Can be shape (N, 3) for N points in 3D, or any shape compatible with box.
    box : float or np.ndarray
        Simulation box dimensions. Can be a scalar (for cubic box) or array-like of shape (3,) for orthorhombic box.

    Returns
    -------
    wrapped_points : np.ndarray
        Array of wrapped points with the same shape as input, all coordinates in [0, box_length).
    """
    box = np.array(box, dtype=float)
    return points % box


def nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False):
    """
    Find the k nearest neighbors of each molecule, based on centroid distance.
    
    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes.
    box : float or array-like
        Simulation box length (for PBC). Pass a scalar if cubic.
    k : int
        Number of nearest neighbors to return per molecule.

    Returns
    -------
    dict[int, list[tuple[int,float]]]
        Mapping mol_id -> list of (neighbor_id, distance).
    """

    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = wrap_points(coords, box)
    kd = cKDTree(coords_wrapped, boxsize=box)

    neighbors = {}
    for idx, mol_id in enumerate(ids):
        dists, idxs = kd.query(coords[idx], k=k+1)
        dists, idxs = dists[1:], idxs[1:]
        if return_meshes:
            neighbors[mol_id] = [(mol_meshes[ids[j]], float(d)) for j, d in zip(idxs, dists)]
        else:
            neighbors[mol_id] = [(ids[j], float(d)) for j, d in zip(idxs, dists)]
    return neighbors


In [27]:
centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box)

neighbors = nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False)

neighbor_candidates = [(key, t[0]) for key, value in neighbors.items() for t in value]

In [28]:
neighbor_candidates_sorted = list(set([(min(key, t[0]), max(key, t[0])) 
                                       for key, value in neighbors.items() for t in value if key < t[0]]))

neighbor_candidates_sorted.sort()

### Blocking algorithm

In [12]:
def blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin=2.0):
    """
    Check if the direct path between two molecule centroids is blocked by any other molecule.

    For a given pair of molecules (i, j), this function determines whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two stages:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Expensive ray-mesh intersection: If the sphere check passes, a ray-mesh intersection
         test is performed to check if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are taken into account using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    kd : scipy.spatial.cKDTree
        KD-tree built from centroid coordinates for efficient neighbor search.
    ids : list[int]
        List of molecule IDs, ordered as in the KD-tree.
    box : np.ndarray
        Simulation box dimensions (in Å).
    cutoff_margin : float, optional
        Additional margin added to the search radius for candidate blockers (default: 2.0 Å).

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len
    midpt = (ci + cj) / 2.0

    # Query potential blockers (MODIFY THIS PART)
    search_radius = seg_len / 2 + max(radii.values()) + cutoff_margin
    cand_idx = kd.query_ball_point(midpt, r=search_radius)

    for kdx in cand_idx:
        mol_k = ids[kdx]   # map index back to mol_id
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [13]:
def find_neighbors(centroids, radii, mol_meshes, box, neighbor_candidates):
    neighbor_pairs = []
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    For each candidate pair (i, j), this function checks if the direct path between the two
    molecule centroids is blocked by any other molecule (using a fast sphere check and an
    expensive ray-mesh intersection test). Only pairs that are not blocked by any other
    molecule are included in the returned neighbor list.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    box : np.ndarray
        Simulation box dimensions (in Å).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """
    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = wrap_points(coords, box)
    kd = cKDTree(coords_wrapped, boxsize=box)

    neighbor_pairs = []
    for pair in tqdm(neighbor_candidates, desc="Testing neighbor pairs"):
        i, j = pair
        if not blocked_by_any(i, j, centroids, radii, mol_meshes, kd, ids, box, cutoff_margin=2.0):
            neighbor_pairs.append((i, j))
            
    return neighbor_pairs

In [14]:
results = find_neighbors(centroids, radii, mol_meshes, box, neighbor_candidates_sorted)

Testing neighbor pairs: 100%|██████████| 7511/7511 [05:35<00:00, 22.39it/s]


`blocked_by_any` and `find_neighbors` algorithm takes about 7.5 minutes for 1501 molecules for k=10.

In [15]:
print("Number of unblocked pairs:",len(results))

Number of unblocked pairs: 1526


### Blocking algorithms 2

In [ ]:
def blocked_by_any_2(i, j, centroids, radii, mol_meshes, ids):
    """
    Determine if the direct path between two molecule centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    ids : list[int]
        List of molecule IDs.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # Map molecule IDs to their index in ids

    ci, cj = centroids[i], centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [36]:
def find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates):
    neighbor_pairs = []
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    box : np.ndarray
        Simulation box dimensions (in Å).
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """

    ids = list(centroids.keys())
    # coords = np.vstack([centroids[i] for i in ids])
    # coords_wrapped = wrap_points(coords, box)
    # kd = cKDTree(coords_wrapped, boxsize=box)

    neighbor_pairs = []
    for pair in tqdm(neighbor_candidates, desc="Testing neighbor pairs"):
        i, j = pair
        if not blocked_by_any_2(i, j, centroids, radii, mol_meshes, ids):
            neighbor_pairs.append((i, j))
            
    return neighbor_pairs

In [39]:
results_2 = find_neighbors_2(centroids, radii, mol_meshes, box, neighbor_candidates_sorted)

Testing neighbor pairs: 100%|██████████| 7511/7511 [03:52<00:00, 32.24it/s] 


`blocked_by_any_2` and `find_neighbors_2` algorithm takes about 3.5 minutes for 1501 molecules for k=10.

In [40]:
print("Number of unblocked pairs:",len(results_2))

Number of unblocked pairs: 3597


### Save the results

In [4]:
import csv

# Save to csv
with open("../../data/processed/unblocked_pairs_1.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

# with open("../../data/processed/unblocked_pairs_2.csv", "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerows(results_2)

NameError: name 'results' is not defined

In [5]:
# Load back
with open("../../data/processed/unblocked_pairs_1.csv") as f:
    reader = csv.reader(f)
    results_1 = [tuple(map(int, row)) for row in reader]

with open("../../data/processed/unblocked_pairs_2.csv") as f:
    reader = csv.reader(f)
    results_2 = [tuple(map(int, row)) for row in reader]

In [6]:
def neighbor_ids(i):
    l1 = [t[1] for t in results_1 if t[0] == i]
    l2 = [t[1] for t in results_2 if t[0] == i]
    print(l1,l2)

    return l1, l2

In [7]:
l1, l2 = neighbor_ids(290)
submesh = [mol_meshes[nid] for nid in l2]
scene = trimesh.Scene(submesh)
scene.show()

[] [696, 794, 1120, 1125, 1489]


NameError: name 'mol_meshes' is not defined

In [15]:
submesh = [mol_meshes[nid] for nid in [207, 254, 311, 372,846,868,919,1238,1368,1379]]
scene = trimesh.Scene(submesh)
scene.show()